*Módulo 6 de 9*

> **Prefer English?** Open [`06_features_and_labels.ipynb`](../en/06_features_and_labels.ipynb) — it is the same module, in English.


# 📊 Módulo 6 — Las parcelas se vuelven tabla + la verdad de campo

🧭 **Objetivos** — convertir cada parcela del Módulo 5 en una **fila de
números** (variables) usando **estadística zonal**, y luego pegarle
etiquetas de **verdad de campo** desde puntos reales con un **filtro de
pureza**. Al final tendrás justo lo que come un clasificador: una tabla de
parcelas × variables, algunas etiquetadas.

📚 **Variables = una parcela como números.** Un modelo no puede mirar una
imagen; necesita números. Para cada parcela resumimos las 13 capas debajo de
ella en **estadística zonal** — aquí la **media** y la **desviación
estándar** de cada banda. Así cada parcela se vuelve una fila de 26 números.
(El pipeline de producción calcula muchas más — mín, máx, sumas, en varios
meses — cientos de columnas; Módulo 9.)

📚 **Verdad de campo = la hoja de respuestas.** Para entrenar un modelo
necesitamos parcelas cuyo cultivo *sí conocemos*. Vienen de **puntos de
campo**: 1,645 ubicaciones GPS del Valle del Yaqui donde alguien registró el
cultivo real. Dejamos caer cada punto en su parcela. Un **filtro de pureza**
conserva solo las parcelas donde todos los puntos coinciden — las parcelas
mixtas son ambiguas y le enseñarían lecciones erróneas al modelo.

![variables](../../anim/es/06_features.svg)

![etiquetas y pureza](../../anim/es/07_labels_purity.svg)


## Reconstruye la segmentación

Cada módulo del curso corre en un kernel limpio, así que volvemos a cargar el
tile y a segmentarlo (mismos parámetros que el Módulo 5) antes de extraer las
variables.


In [ ]:
# Trae el tile del taller (pocos MB; queda en caché tras la primera descarga)
import os, sys

async def trae_archivo(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await trae_archivo("crop_tile_384.tif")
print("Tile listo:", TILE)

In [ ]:
import numpy as np, rasterio
import scipy.ndimage, sklearn.cluster
import shepherd_wasm

with rasterio.open(TILE) as src:
    img = src.read()
    nombres_banda = list(src.descriptions)

resultado = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0, fixedKMeansInit=True)
seg = resultado.segimg.astype(np.int32)
n_seg = int(seg.max())
print(f"{n_seg} parcelas por describir")

## Estadística zonal con np.bincount

Para cada banda queremos la media y la desviación estándar de los píxeles
dentro de cada parcela. `np.bincount` lo hace rápido: suma valores agrupados
por id de segmento. De la suma y la suma de cuadrados obtenemos la media y la
desviación de todas las parcelas a la vez — vectorizado, sin ciclo de Python
por parcela.


In [ ]:
seg_plano = seg.ravel()
conteos = np.bincount(seg_plano, minlength=n_seg + 1).astype(float)
conteos[conteos == 0] = 1     # evita dividir entre 0 para ids sin uso

variables = np.zeros((n_seg + 1, len(nombres_banda) * 2), dtype=np.float32)
for b in range(len(nombres_banda)):
    vals = img[b].ravel().astype(np.float64)
    s1 = np.bincount(seg_plano, weights=vals,       minlength=n_seg + 1)
    s2 = np.bincount(seg_plano, weights=vals * vals, minlength=n_seg + 1)
    media = s1 / conteos
    var   = np.maximum(s2 / conteos - media**2, 0)
    variables[:, 2*b], variables[:, 2*b+1] = media, np.sqrt(var)

nombres_variables = [f"{n}_{s}" for n in nombres_banda for s in ("media", "desv")]
print(f"Tabla de variables: {variables.shape[0]-1} parcelas x {variables.shape[1]} variables")
print("Primeros nombres de variable:", nombres_variables[:4])

## Pega la verdad de campo (con el filtro de pureza)

El raster de etiquetas trae el id de cultivo real en las ubicaciones de los
puntos de campo. Para cada parcela que contiene píxeles etiquetados, la
conservamos para entrenar **solo si todos los píxeles etiquetados de adentro
coinciden** en la misma clase — ese es el filtro de pureza. Luego contamos
cuántas parcelas puras tenemos por cultivo.


In [ ]:
import json

async def trae_archivo(name):
    import os, sys
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url); open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request; urllib.request.urlretrieve(url, dest)
    return dest

LABELS  = await trae_archivo("crop_labels_384.tif")
NOMBRES = await trae_archivo("class_names.json")

with rasterio.open(LABELS) as src:
    lab = src.read(1)
nombres_clase = {int(k): v for k, v in json.load(open(NOMBRES)).items()}

etiqueta_parcela = np.zeros(n_seg + 1, dtype=int)
for sid in np.unique(seg[lab > 0]):
    valores = lab[(seg == sid) & (lab > 0)]
    unicos = np.unique(valores)
    if len(unicos) == 1:               # parcela pura -> sirve para entrenar
        etiqueta_parcela[sid] = unicos[0]

ids_entrena = np.flatnonzero(etiqueta_parcela)
print(f"Parcelas etiquetadas puras: {len(ids_entrena)} de {n_seg}")
for cid, cname in nombres_clase.items():
    print(f"  {cname:12s}: {(etiqueta_parcela[ids_entrena] == cid).sum():3d} parcelas")

## 🧪 Ponte a prueba

**¿Por qué resumir cada parcela a una media y una desviación en vez de darle
al modelo todos sus píxeles?**

<details><summary>Ver respuesta</summary>

El modelo clasifica *parcelas*, y necesita una fila de números de largo fijo
por parcela — pero las parcelas tienen distinto número de píxeles. La
estadística zonal (media, desv, ...) comprime cualquier parcela en el mismo
conjunto de variables, y captura lo que importa: el valor típico de la
parcela y qué tan uniforme es.

</details>

**¿Qué desecha el filtro de pureza, y por qué eso es bueno para el
entrenamiento?**

<details><summary>Ver respuesta</summary>

Descarta parcelas cuyos puntos de campo no coinciden en el cultivo (parcelas
mixtas o mal etiquetadas). Entrenar solo con parcelas de una única clase
acordada evita enseñarle al modelo ejemplos contradictorios, que
difuminarían cada clase que aprende.

</details>


## 🔭 Profundiza

Opcional: estas tarjetas bilingües de conceptos amplían lo que acabas
de aprender (prerrequisitos, linaje a fundamentos, referencias):

- [Estadística zonal](https://abxda.github.io/rs-learning-audio/?id=zonal-statistics&lang=es)
- [Verdad de campo](https://abxda.github.io/rs-learning-audio/?id=ground-truth&lang=es)
- [Muestras de entrenamiento](https://abxda.github.io/rs-learning-audio/?id=training-samples&lang=es)
- [Balanceo de muestras](https://abxda.github.io/rs-learning-audio/?id=sample-balancing&lang=es)



---

[← Anterior · Módulo 5 — De píxeles a parcelas: segmentación](05_de_pixeles_a_parcelas.ipynb) · [Siguiente → · Módulo 7 — Aprendizaje automático desde cero](07_machine_learning_desde_cero.ipynb)
